# LangChain Expression Language (LCEL)
### *Composing intelligent LLM pipelines with a clean, chainable syntax*

---

> **LCEL** is LangChain's declarative way to build workflows by chaining components together using the `|` (pipe) operator — just like Unix pipes, but for AI.

---

## What we build in this notebook

```
PromptTemplate  →  OpenAI LLM  →  StrOutputParser
```

A simple but complete **LCEL chain** that:
1. Formats a dynamic prompt with user-provided variables
2. Sends it to a GPT model
3. Returns a clean string response

---

## Step 1 — Install Dependencies

Install the three packages needed to run this notebook:

| Package | Role |
|---------|------|
| `langchain` | Core framework — chains, prompts, parsers |
| `langchain-openai` | OpenAI wrapper for GPT models |

In [1]:
!pip install langchain langchain-openai python-dotenv

INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0rc2
    Uninstalling packaging-26.0rc2:
      Successfully uninstalled packaging-26.0rc2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

## Step 2 — Import Libraries

| Import | Purpose |
|--------|---------|
| `os` | Access environment variables |
| `load_dotenv` | Load `.env` file into the environment |
| `OpenAI` | Call GPT models via LangChain |
| `PromptTemplate` | Define structured prompts with placeholders |
| `StrOutputParser` | Parse the LLM output as a clean Python string |

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Step 3 — Load the API Key

The OpenAI API key is retrieved securely and injected as an environment variable.  


In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key  = user_secrets.get_secret("OPENAI_API_KEY")

## Step 4 — Initialize the LLM

We instantiate the OpenAI model with two key parameters:

| Parameter | Value | Effect |
|-----------|-------|--------|
| `temperature` | `0.7` | Balanced creativity — not too rigid, not too random |
| `openai_api_key` | from env | Authenticates the API call |

> `temperature=0` → fully deterministic · `temperature=1` → highly creative

In [6]:
llm = OpenAI( 
    temperature=0.7,
    openai_api_key=api_key
)

## Step 5 — Run a Simple Prompt

A direct call to the LLM with a raw string prompt — no template, no chain.  
This is the most basic usage to verify the setup is working correctly.

In [7]:
prompt = "Suggest me a skill that is in demand?"
response = llm.invoke(prompt)
print(" Suggested Skill:\n", response)

 Suggested Skill:
 

Digital marketing is a skill that is currently in high demand. With the rise of internet usage and social media, businesses are seeking professionals who can effectively promote their products and services online. This includes skills such as search engine optimization (SEO), social media management, content creation, and data analysis. Digital marketing skills can be applied to a wide range of industries and can lead to lucrative job opportunities. 


## Step 6 — Create a Prompt Template

A `PromptTemplate` lets you **parameterize** your prompts with placeholders like `{year}`.  
At runtime, `.invoke({"year": "2025"})` fills in the blank automatically.

In [8]:
template = "Give me 3 career skills that are in high demand in {year}."
prompt_template = PromptTemplate.from_template(template)

## Step 7 — Build the LCEL Chain

The `|` operator **pipes the output of each component into the next**, creating a fully composable pipeline:

```
PromptTemplate  ──|──►  LLM  ──|──►  StrOutputParser
  fills {year}       generates text     returns clean str
```

| Component | Input | Output |
|-----------|-------|--------|
| `prompt_template` | `{"year": "2025"}` | Formatted prompt string |
| `llm` | Formatted prompt | Raw model response |
| `StrOutputParser()` | Raw response | Clean Python `str` |


LCEL (LangChain Expression Language) : C'est une nouvelle façon de composer des flux de travail LLM en utilisant une syntaxe simple et chaînable avec l'opérateur | (pipe).

1. modèle_invite

Remplit les espaces réservés (comme {année}) avec les entrées réelles.
Exemple : « Donnez-moi 3 compétences professionnelles en 2025. »

2. llm

Envoie l'invite formatée au modèle OpenAI.
Exemple de saisie : « Donnez-moi 3 compétences professionnelles en 2025. »
Exemple de résultat : « 1. Analyse de données\n2. IA/ML\n3. Cybersécurité »

3. StrOutputParser()

Nettoie et garantit que la réponse du LLM est renvoyée sous forme de chaîne de caractères.

In [9]:
chain = prompt_template | llm | StrOutputParser()

## Step 8 — Run the Chain

`.invoke({"year": "2025"})` triggers the full pipeline end-to-end:  
the placeholder is filled → the prompt is sent to GPT → the result is parsed and printed.

> Change `"2025"` to any year to instantly get a different, context-aware response.

In [10]:
response = chain.invoke({"year": "2025"})
print("\n Career Skills in 2025:\n", response)


 Career Skills in 2025:
 

1. Data analysis and interpretation: With the increasing use of technology and data in various industries, the ability to collect, analyze, and interpret data will be in high demand. This skill will help businesses make data-driven decisions and improve their overall performance.

2. Artificial Intelligence and machine learning: As AI and machine learning continue to advance, there will be a growing need for professionals who can develop, maintain, and implement these technologies. This skill will be valuable in fields such as healthcare, finance, and manufacturing.

3. Digital marketing: With the rise of e-commerce and online platforms, digital marketing skills will be in high demand in 2025. Companies will need professionals who can create effective online marketing strategies, manage social media platforms, and analyze digital marketing data to reach and engage their target audience.
